# File loading

In [1]:
import pandas as pd

df = pd.read_csv("fixation/18sat_fixfinal.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.dtypes)

(463564, 29)
['RECORDING_SESSION_LABEL', 'TRIAL_INDEX', 'CURRENT_FIX_X', 'CURRENT_FIX_Y', 'CURRENT_FIX_PUPIL', 'CURRENT_FIX_DURATION', 'CURRENT_FIX_INTEREST_AREA_ID', 'CURRENT_FIX_INTEREST_AREA_LABEL', 'CURRENT_FIX_INTEREST_AREA_PIXEL_AREA', 'CURRENT_FIX_INTEREST_AREA_RUN_ID', 'CURRENT_FIX_INTEREST_AREA_DWELL_TIME', 'PREVIOUS_SAC_DIRECTION', 'PREVIOUS_SAC_ANGLE', 'PREVIOUS_SAC_AMPLITUDE', 'PREVIOUS_SAC_AVG_VELOCITY', 'PREVIOUS_SAC_CONTAINS_BLINK', 'PREVIOUS_SAC_BLINK_DURATION', 'Session_Name_', 'Trial_Index_', 'Trial_Recycled_', 'total_page', 'type', 'book_name', 'book', 'page', 'RT', 'answer', 'correct_answer', 'page_name']
  RECORDING_SESSION_LABEL  TRIAL_INDEX  CURRENT_FIX_X  CURRENT_FIX_Y  \
0                  msd001            1           59.8          125.4   
1                  msd001            1          348.7          182.0   
2                  msd001            1          630.5          400.3   
3                  msd001            1          492.0          400.2   
4      

In [2]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 463564 entries, 0 to 463563
Data columns (total 29 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   RECORDING_SESSION_LABEL               463564 non-null  str    
 1   TRIAL_INDEX                           463564 non-null  int64  
 2   CURRENT_FIX_X                         463564 non-null  float64
 3   CURRENT_FIX_Y                         463564 non-null  float64
 4   CURRENT_FIX_PUPIL                     463564 non-null  float64
 5   CURRENT_FIX_DURATION                  463564 non-null  int64  
 6   CURRENT_FIX_INTEREST_AREA_ID          251087 non-null  float64
 7   CURRENT_FIX_INTEREST_AREA_LABEL       251087 non-null  str    
 8   CURRENT_FIX_INTEREST_AREA_PIXEL_AREA  251087 non-null  float64
 9   CURRENT_FIX_INTEREST_AREA_RUN_ID      251087 non-null  float64
 10  CURRENT_FIX_INTEREST_AREA_DWELL_TIME  251087 non-null  float64
 11  PREVIOUS_SA

# Pre-processing

In [3]:
## Distinguishing between reading the passages or answering the questions

reading = df[df["type"] == "reading"].copy()
questions = df[df["type"] == "question"].copy()

In [4]:
## Get comprehension score per passage
# One row per question rather than one row per fixation
question_trials = (
    questions[
        [
            "RECORDING_SESSION_LABEL",
            "book_name",
            "page",
            "answer",
            "correct_answer"
        ]
    ]
    .drop_duplicates()
)

# Scored comprehension questions only
comprehension_questions = question_trials[
    question_trials["correct_answer"] != -99
].copy()

comprehension_questions["correct"] = (
    comprehension_questions["answer"]
    == comprehension_questions["correct_answer"]
).astype(int)

passage_comprehension = (
    comprehension_questions
    .groupby(["RECORDING_SESSION_LABEL", "book_name"])
    .agg(
        n_correct=("correct", "sum"),
        n_questions=("correct", "count"),
        comprehension_score=("correct", "mean")
    )
    .reset_index()
)

In [5]:
print(passage_comprehension.head())

  RECORDING_SESSION_LABEL  book_name  n_correct  n_questions  \
0                  msd001    dickens          3            5   
1                  msd001    flytrap          4            5   
2                  msd001     genome          4            5   
3                  msd001  northpole          4            5   
4                  msd002    dickens          3            5   

   comprehension_score  
0                  0.6  
1                  0.8  
2                  0.8  
3                  0.8  
4                  0.6  


In [6]:
## Try to get the same median score as paper to classify as high/low comprehension level

median_comprehension = passage_comprehension["comprehension_score"].median()

print("Median comprehension:", median_comprehension)


Median comprehension: 0.6


In [7]:
passage_comprehension["comprehension_level"] = (
    passage_comprehension["comprehension_score"]
    .apply(lambda x: "High" if x >= median_comprehension else "Low")
)

print(passage_comprehension.head(10))

  RECORDING_SESSION_LABEL  book_name  n_correct  n_questions  \
0                  msd001    dickens          3            5   
1                  msd001    flytrap          4            5   
2                  msd001     genome          4            5   
3                  msd001  northpole          4            5   
4                  msd002    dickens          3            5   
5                  msd002    flytrap          0            5   
6                  msd002     genome          3            5   
7                  msd002  northpole          3            5   
8                  msd003    dickens          3            5   
9                  msd003    flytrap          4            5   

   comprehension_score comprehension_level  
0                  0.6                High  
1                  0.8                High  
2                  0.8                High  
3                  0.8                High  
4                  0.6                High  
5                  0.0   

In [8]:
passage_comprehension.shape

(380, 6)

In [9]:
## Calculate overall_comprehension

overall_comprehension = (
    passage_comprehension
    .groupby("RECORDING_SESSION_LABEL")
    .agg(
        n_correct=("n_correct", "sum"),
        n_questions=("n_questions", "sum")
    )
    .reset_index()
)

overall_comprehension["comprehension_score"] = (
    overall_comprehension["n_correct"]
    / overall_comprehension["n_questions"]
)

print(overall_comprehension.head())

  RECORDING_SESSION_LABEL  n_correct  n_questions  comprehension_score
0                  msd001         15           20                 0.75
1                  msd002          9           20                 0.45
2                  msd003         13           20                 0.65
3                  msd004         12           20                 0.60
4                  msd005         15           20                 0.75


In [10]:
overall_median = overall_comprehension["comprehension_score"].median()

print("Overall comprehension median:", overall_median)

Overall comprehension median: 0.55


In [11]:
overall_comprehension["comprehension_level"] = (
   overall_comprehension["comprehension_score"]
    .apply(lambda x: "High" if x >= overall_median else "Low")
)

print(overall_comprehension.head(10))


  RECORDING_SESSION_LABEL  n_correct  n_questions  comprehension_score  \
0                  msd001         15           20                 0.75   
1                  msd002          9           20                 0.45   
2                  msd003         13           20                 0.65   
3                  msd004         12           20                 0.60   
4                  msd005         15           20                 0.75   
5                  msd007         17           20                 0.85   
6                  msd008         11           20                 0.55   
7                  msd009         12           20                 0.60   
8                  msd010          9           20                 0.45   
9                  msd011         10           20                 0.50   

  comprehension_level  
0                High  
1                 Low  
2                High  
3                High  
4                High  
5                High  
6                

In [12]:
overall_comprehension.shape

(95, 5)